# Notebook 3 — Predictions and Refusals
**YTS+ DSEB 2026 · Project A · Harry Potter**

---

In Notebooks 1 and 2 you used the network to describe the story: who appears where, which communities form, who leads each world.

Now you will use it to make **predictions** — about events that had already happened before you ran any code.

The logic: two characters who share many mutual acquaintances inhabit the same narrative world. They attend the same places, know the same people, move in the same circles. In a realistic social network, those two people would almost certainly meet. In a Rowling novel — they usually do. But not always.

The cases where the network says *they should have met* and Rowling chose *they never did* — those are the most interesting result in this project. They are not accidents. They reveal something deliberate about how Rowling organised her story worlds.

**The final question:** Which refused meeting would you write into the films — and why does the network say it should exist?

---
## Setup

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import time
import os

os.makedirs('images', exist_ok=True)
DATA = 'data/'
print('Ready.')

---
## Part 1 — EDA: Common neighbors after Book 3

**Common neighbors** (CN) counts how many characters two people both appear with. If Harry and Draco both appear with Snape, Dumbledore, and McGonagall — their CN is at least 3.

High CN means two characters inhabit the same narrative world. The question is: does Rowling eventually put them in a scene together?

In [ ]:
cn = pd.read_csv(DATA + 'common_neighbors_book3.csv')

print(f'Shape: {cn.shape}')
print(f'Columns: {cn.columns.tolist()}')
print()
cn.head(10)

In [ ]:
print('Common neighbor count distribution:')
print(cn['common_neighbors'].describe())
print()
print(f'Pairs with CN >= 5:  {(cn["common_neighbors"] >= 5).sum()}')
print(f'Pairs with CN >= 10: {(cn["common_neighbors"] >= 10).sum()}')
print(f'Pairs with CN >= 15: {(cn["common_neighbors"] >= 15).sum()}')
print(f'Pairs with CN >= 20: {(cn["common_neighbors"] >= 20).sum()}')

In [ ]:
total = len(cn)
ever_meet = cn['ever_meet'].sum()
never_meet = total - ever_meet

print(f'Total pairs (CN >= 3): {total}')
print(f'  Meet somewhere in Books 1-7: {ever_meet} ({ever_meet/total*100:.0f}%)')
print(f'  Never meet:                  {never_meet} ({never_meet/total*100:.0f}%)')
print()

print('Meet rate by CN threshold:')
for threshold in [5, 10, 15, 20]:
    sub = cn[cn['common_neighbors'] >= threshold]
    rate = sub['ever_meet'].mean()
    print(f'  CN >= {threshold:>2}: {len(sub):>5} pairs | {rate*100:.0f}% eventually meet')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

meet = cn[cn['ever_meet'] == True]['common_neighbors']
no_meet = cn[cn['ever_meet'] == False]['common_neighbors']

ax.hist(no_meet, bins=30, color='tomato', alpha=0.7, label='Never meet', edgecolor='white')
ax.hist(meet, bins=30, color='steelblue', alpha=0.7, label='Eventually meet', edgecolor='white')
ax.set_xlabel('Common neighbors after Book 3')
ax.set_ylabel('Number of pairs')
ax.set_title('Common neighbors after Book 3 — do they eventually meet?')
ax.legend()
plt.tight_layout()
plt.savefig('images/cn_distribution.png', dpi=150)
plt.show()

In [ ]:
meet_pairs = cn[cn['ever_meet'] == True].copy()
book_counts = meet_pairs['first_meet_book'].value_counts().sort_index()

print('When do high-CN pairs first meet?')
print(book_counts.to_string())

---
## Part 2 — EDA: Never-meeting pairs

In [ ]:
nm = pd.read_csv(DATA + 'never_meeting_pairs.csv')

print(f'Shape: {nm.shape}')
print(f'Columns: {nm.columns.tolist()}')
print()
nm.head(8)

In [ ]:
print('Never-meeting pairs breakdown:')
print(f'  Total pairs (CN >= 15 after Book 5): {len(nm)}')
print(f'  Dead character:       {nm["is_dead_character"].sum()}')
print(f'  Historical figure:    {nm["is_historical_figure"].sum()}')
print(f'  Narrative exit:       {nm["is_narrative_exit"].sum()}')
print(f'  Genuine refusals:     {nm["genuine_refusal"].sum()}')
print()

genuine = nm[nm['genuine_refusal'] == True].sort_values(
    'common_neighbors_after_book5', ascending=False)
print(f'Top genuine refusals:')
print(genuine[['char_a', 'char_b', 'common_neighbors_after_book5']].head(10).to_string(index=False))

In [ ]:
genuine = nm[nm['genuine_refusal'] == True]
explained = nm[nm['genuine_refusal'] == False]

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(explained['common_neighbors_after_book5'], bins=20, color='gray', alpha=0.6,
        label='Explained (dead / exited)', edgecolor='white')
ax.hist(genuine['common_neighbors_after_book5'], bins=20, color='firebrick', alpha=0.7,
        label='Genuine refusal', edgecolor='white')
ax.set_xlabel('Common neighbors after Book 5')
ax.set_ylabel('Number of pairs')
ax.set_title('Never-meeting pairs — CN after Book 5')
ax.legend()
plt.tight_layout()
plt.savefig('images/genuine_refusals_cn.png', dpi=150)
plt.show()

---
## Part 3 — Your predictions

Write now, before searching the data:
- **2 pairs** you predict will meet in Book 4 (haven't yet after Book 3)
- **2 pairs** you predict will never meet across all 7 books

Use your knowledge of the story — not the data.

**Write here:**

Will meet in Book 4:
1. 
2. 

Will never meet:
1. 
2. 

In [ ]:
def lookup_pair(a, b):
    match = cn[((cn['char_a'] == a) & (cn['char_b'] == b)) |
               ((cn['char_a'] == b) & (cn['char_b'] == a))]
    if len(match) == 0:
        print(f'{a} / {b}: CN < 3 (barely share mutual characters)')
    else:
        row = match.iloc[0]
        print(f'{a} / {b}:')
        print(f'  Common neighbors: {int(row["common_neighbors"])}')
        print(f'  Ever meet: {row["ever_meet"]}')
        if row['ever_meet']:
            print(f'  First meet: Book {int(row["first_meet_book"])}')

# Replace with your predicted pairs
lookup_pair('Harry Potter', 'Draco Malfoy')        # example
print()
lookup_pair('Gilderoy Lockhart', 'Sirius Black')   # known interesting failure

**For each pair that the network predicted but never met — which explanation applies?**

- The character died, was obliviated, or otherwise exited the story
- The character graduated, left Hogwarts, or moved to a different world
- Rowling made a deliberate choice to keep them apart even though the structure suggested otherwise

That last category is the one that matters. Name any pair from your predictions or the top refusals where you think the explanation is deliberate authorial choice — not circumstance.

**Write here:**

*Write here.*

---
## Part 4 — How good is the prediction?

Before you look at the number — think about this: some of those shared mutual characters might come from crowd sentences. When Rowling writes *"Seamus, Dean, Neville, and Lavender all cheered,"* the algorithm creates an edge between every pair in that list. Does Seamus genuinely have a connection to Lavender — or were they just named in the same sentence? Keep that in mind when you decide how impressed to be by the precision.

In [ ]:
print(f'{"CN threshold":<15} {"Pairs predicted":>16} {"Actually met":>13} {"Precision":>10}')
print('-' * 58)

for threshold in [5, 8, 10, 12, 15, 20]:
    predicted = cn[cn['common_neighbors'] >= threshold]
    met = predicted['ever_meet'].sum()
    precision = met / len(predicted) if len(predicted) > 0 else 0
    print(f'CN >= {threshold:<9} {len(predicted):>16} {int(met):>13} {precision*100:>9.0f}%')

In [ ]:
thresholds = list(range(3, 25))
precisions = []
pair_counts = []

for t in thresholds:
    sub = cn[cn['common_neighbors'] >= t]
    precisions.append(sub['ever_meet'].mean() if len(sub) > 0 else 0)
    pair_counts.append(len(sub))

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

ax1.plot(thresholds, [p*100 for p in precisions], 'o-', color='steelblue', linewidth=2, label='Precision (%)')
ax2.bar(thresholds, pair_counts, alpha=0.2, color='gray', label='Pairs predicted')

ax1.axhline(92, color='steelblue', linestyle='--', alpha=0.5)
ax1.text(22, 93, '92%', color='steelblue', fontsize=9)

ax1.set_xlabel('Common neighbor threshold')
ax1.set_ylabel('Precision (% who actually meet)', color='steelblue')
ax2.set_ylabel('Number of pairs predicted', color='gray')
ax1.set_title('Link prediction — precision vs threshold')
plt.tight_layout()
plt.savefig('images/precision_curve.png', dpi=150)
plt.show()

**CN > 10 after Book 3 gives 92% precision.** The algorithm correctly predicted 92% of high-CN meetings using nothing but proximity counts from the first three books.

**Write:** The 8% that failed — is that randomness, or do you see a pattern in which pairs the algorithm got wrong? What does Rowling do that the network cannot anticipate?

**Your answer:**

*Write here.*

---
## Part 5 — The genuine refusals

In [ ]:
genuine = nm[nm['genuine_refusal'] == True].sort_values(
    'common_neighbors_after_book5', ascending=False).reset_index(drop=True)

print(f'Genuine refusals: {len(genuine)} pairs\n')
print(genuine[['char_a', 'char_b', 'common_neighbors_after_book5']].head(12).to_string(index=False))

In [ ]:
top8 = genuine.head(8)
labels = [f"{row['char_a'].split()[-1]} –\n{row['char_b'].split()[-1]}"
          for _, row in top8.iterrows()]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(labels[::-1], top8['common_neighbors_after_book5'].tolist()[::-1], color='firebrick')
ax.set_xlabel('Common neighbors after Book 5')
ax.set_title('Genuine refusals — high-CN pairs who never share a scene')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('images/genuine_refusals_bar.png', dpi=150)
plt.show()

**These are not accidents or oversights.**

Each of these pairs is alive, active in the story, and sharing 29–40 mutual characters. A physics-based model says: with that much shared social space, a collision is statistically inevitable. Rowling said no.

That no is a decision. It reveals how she organized the boundaries between her story worlds.

Read the top 5 refusals. For each one, ask: **what narrative boundary does Rowling maintain by keeping them apart?**

Some cases to think about:
- **McGonagall — Cho Chang (CN=40):** Both embedded in Hogwarts. Both deeply present in Books 5–7. What worlds do they each belong to — and why would Rowling not bridge them?
- **Filch — Tom Riddle (CN=31):** The caretaker and the Dark Lord. What does it mean that Rowling kept the mundane and the supernatural completely separate?
- **Fred Weasley — Seamus Finnigan (CN=31):** Both Gryffindors, both prominent. No scene together in the entire series.

**Write:** Pick one refusal from the list. In 3–4 sentences, explain what you think Rowling knew that the network didn't — and what the network found that you wouldn't have noticed from reading.

**Your answer:**

*Write here.*

---
## Part 6 — The screenwriter question

You are adapting Books 5–7 into films. You must add one scene involving a never-meeting pair.

First, find the mutual characters for your chosen pair — they show you what world the two characters share, which tells you where the scene could plausibly happen.

In [ ]:
t0 = time.time()

dfs = [pd.read_csv(DATA + f'hp_book{i}_edges.csv') for i in range(1, 6)]
df_all5 = pd.concat(dfs).groupby(['source', 'target'])['weight'].sum().reset_index()
G_all5 = nx.from_pandas_edgelist(df_all5, 'source', 'target', edge_attr='weight')

print(f'Cumulative Books 1-5: {G_all5.number_of_nodes()} characters, {G_all5.number_of_edges()} pairs | {time.time()-t0:.2f}s')

In [ ]:
# Change these to your chosen pair
char_a = 'Minerva McGonagall'
char_b = 'Cho Chang'

mutual = list(nx.common_neighbors(G_all5, char_a, char_b))
mutual_sorted = sorted(mutual, key=lambda n: G_all5.degree(n, weight='weight'), reverse=True)

print(f'{char_a} and {char_b} share {len(mutual)} mutual characters:\n')
print(', '.join(mutual_sorted[:15]))

**Your 60-second screenwriter pitch:**

**Pair:**

**CN score:**

**3 mutual characters — what world do they share:**
1. 
2. 
3. 

**The scene** — where, what happens, what do they say:

*Write here.*

**Why Rowling kept them apart — what boundary was she maintaining:**

*Write here.*

**Why a film would add it — what does the scene unlock:**

*Write here.*

---
## Closing — what did the instrument find?

Open your sealed prediction from Notebook 1.

> **You predicted: if Harry were removed, ___ becomes the most important character.**

Look at what the within-community rankings (Notebook 2) actually showed. Check who led the Order world, the school world, the Slytherin world.

**Write 3–4 sentences:**
- Was your prediction right? Which community result confirms or challenges it?
- Across all three notebooks — name one thing the network found that you would not have noticed from reading. Give the specific number.
- And name one thing the network got wrong, or measured so crudely that the result misled more than it revealed.

This last question is the point of the project. The instrument is not a better reader than you. It is a different reader. What it sees and what it misses together tell you something about what kind of thing a story is.

**Write here:**

*Write here.*